In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio

from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

!pip install --quiet lazypredict
from lazypredict.Supervised import LazyClassifier, LazyRegressor

import warnings
warnings.filterwarnings('ignore')

In [ ]:
soil_nutrients = pd.read_csv('/content/Soil Nutrients.csv')
soil_nutrients

In [ ]:
encoder = LabelEncoder()
label_mappings = {}

for column in soil_nutrients.select_dtypes('object'):
    soil_nutrients[column] = encoder.fit_transform(soil_nutrients[column])
    label_mappings[column] = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

for column, mapping in label_mappings.items():
    print(f"Column: {column}")
    for label, encoding in mapping.items():
        print(f"{encoding} -> {label}")
    print()

In [ ]:
pca = PCA(n_components = 3)
pca_soil_nutrients = pca.fit_transform(soil_nutrients.drop(columns = ['Name', 'Yield']))
pca_soil_nutrients = pd.DataFrame(data = pca_soil_nutrients, columns = ['PCA Component 1', 'PCA Component 2', 'PCA Component 3'])
pca_soil_nutrients

In [ ]:
pca_soil_nutrients['Name'] = soil_nutrients['Name'].map({value: key for key, value in label_mappings['Name'].items()})
pca_soil_nutrients['Yield'] = soil_nutrients['Yield']
pca_soil_nutrients

In [ ]:
pio.renderers.default = 'iframe'
import plotly.io as pio
pio.renderers.default = 'colab'

fig = px.scatter_3d(
    pca_soil_nutrients,
    x = 'PCA Component 1',
    y = 'PCA Component 2',
    z = 'PCA Component 3',
    color = 'Name',
    title = '3D PCA Projection Colored by Crop',
    labels = {
        'PCA Component 1' : 'PCA Component 1',
        'PCA Component 2' : 'PCA Component 2',
        'PCA Component 3' : 'PCA Component 3',
        'Name' : 'Crop'
    }
)

fig.show()

In [ ]:
class CropClassifier:
    def __init__(self, data, features_for_classification, test_size = 0.2, random_state = 42):
        self.data = data
        self.features_for_classification = features_for_classification
        self.test_size = test_size
        self.random_state = random_state

        self.features = self.data.drop(self.features_for_classification, axis = 1)
        self.targets = self.data[self.features_for_classification]
        self.features_train, self.features_test, self.targets_train, self.targets_test = train_test_split(self.features, self.targets, test_size = self.test_size, random_state = self.random_state, stratify = self.targets)

        self.lazyClassifier = LazyClassifier(verbose = 0, ignore_warnings = True, custom_metric = None)
        self.models = None
        self.predictions = None

    def fit(self):
        self.models, self.predictions = self.lazyClassifier.fit(self.features_train, self.features_test, self.targets_train, self.targets_test)

    def get_models(self):
        return self.models

    def get_predictions(self):
        return self.predictions

In [ ]:
cropClassifier = CropClassifier(pca_soil_nutrients, ['Name'])
cropClassifier.fit()

In [ ]:
# ============================================================
# Feature Importance - Exact Recreation
# ============================================================
if best_tree_model:
    model = results_sorted[best_tree_model]['model']

    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        feature_names = X.columns

        # Create DataFrame (Top 20)
        fi_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importances
        }).sort_values('Importance', ascending=True).tail(20)  # Ascending for bottom-to-top

        # Create figure with exact dimensions
        fig, ax = plt.subplots(figsize=(14, 12))

        # Create colormap (yellow to dark blue)
        norm = plt.Normalize(vmin=fi_df['Importance'].min(),
                            vmax=fi_df['Importance'].max())
        colors = plt.cm.plasma(norm(fi_df['Importance']))

        # Create horizontal bars
        bars = ax.barh(fi_df['Feature'], fi_df['Importance'],
                      color=colors, height=0.7)

        # Add importance values on the right
        for i, (feature, importance) in enumerate(zip(fi_df['Feature'], fi_df['Importance'])):
            ax.text(importance + 0.003, i, f'{importance:.3f}',
                   va='center', ha='left', fontsize=13,
                   fontweight='normal', color='black')

        # Styling
        ax.set_xlabel('Feature Importance Score', fontsize=15,
                     fontweight='normal', labelpad=10)
        ax.set_ylabel('Agricultural Feature', fontsize=15,
                     fontweight='normal', labelpad=10)

        # Remove top and right spines
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Font sizes for ticks
        ax.tick_params(axis='both', labelsize=13)

        # Grid
        ax.grid(axis='x', alpha=0.2, linestyle='-', linewidth=0.5)
        ax.set_axisbelow(True)

        # Set x-axis limits
        ax.set_xlim(0, fi_df['Importance'].max() * 1.15)

        # Add colorbar on the right
        sm = plt.cm.ScalarMappable(cmap='plasma', norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, pad=0.08, aspect=30)
        cbar.set_label('Importance', fontsize=14, rotation=270, labelpad=25)
        cbar.ax.tick_params(labelsize=11)

        # Background
        ax.set_facecolor('white')
        fig.patch.set_facecolor('white')

        plt.tight_layout()
        plt.show()

In [ ]:
models = cropClassifier.get_models().style.format(precision = 5)
models

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'colab'

fig = px.scatter_3d(
    pca_soil_nutrients,
    x='PCA Component 1',
    y='PCA Component 2',
    z='PCA Component 3',
    color='Yield',
    title='3D Agricultural Field - Yield Heatmap',
    labels={
        'PCA Component 1': 'Field Length',
        'PCA Component 2': 'Field Width',
        'PCA Component 3': 'Elevation',
        'Yield': 'Crop Yield'
    },
    color_continuous_scale='RdYlGn',  # Red=Low, Green=High
    size_max=10,
    opacity=0.8
)

# Top-down agricultural field view
fig.update_layout(
    scene=dict(
        camera=dict(
            eye=dict(x=0, y=0, z=2.5),  # Bird's eye view
            center=dict(x=0, y=0, z=0)
        ),
        xaxis=dict(showgrid=True, gridcolor='wheat', backgroundcolor='rgb(230, 230, 200)'),
        yaxis=dict(showgrid=True, gridcolor='wheat', backgroundcolor='rgb(230, 230, 200)'),
        zaxis=dict(showgrid=True, gridcolor='wheat', backgroundcolor='rgb(230, 230, 200)'),
        bgcolor='rgb(240, 235, 215)'  # Soil color
    ),
    width=1000,
    height=800
)

fig.update_traces(marker=dict(
    size=4,
    line=dict(width=0.5, color='DarkSlateGrey')
))

fig.show()

In [ ]:
# ============================================================
# CELL 1: Import Libraries
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'colab'

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")


# ============================================================
# CELL 2: 3D PCA Agricultural Field Heatmap
# ============================================================
fig = px.scatter_3d(
    pca_soil_nutrients,
    x='PCA Component 1',
    y='PCA Component 2',
    z='PCA Component 3',
    color='Yield',
    title='<b>3D PCA Analysis: Yield Distribution in Agricultural Space</b>',
    labels={
        'PCA Component 1': 'Principal Component 1 (Soil Properties)',
        'PCA Component 2': 'Principal Component 2 (Climate Factors)',
        'PCA Component 3': 'Principal Component 3 (Vegetation Index)',
        'Yield': 'Crop Yield (kg/ha)'
    },
    color_continuous_scale='RdYlGn',
    opacity=0.7
)

fig.update_layout(
    scene=dict(
        camera=dict(
            eye=dict(x=0, y=0, z=2.5),
            center=dict(x=0, y=0, z=0)
        ),
        xaxis=dict(showgrid=True, gridcolor='lightgray', backgroundcolor='rgb(240, 240, 240)'),
        yaxis=dict(showgrid=True, gridcolor='lightgray', backgroundcolor='rgb(240, 240, 240)'),
        zaxis=dict(showgrid=True, gridcolor='lightgray', backgroundcolor='rgb(240, 240, 240)'),
        bgcolor='rgb(250, 250, 250)'
    ),
    font=dict(family="Arial, sans-serif", size=12),
    width=1000,
    height=700,
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.update_traces(marker=dict(size=3, line=dict(width=0.3, color='white')))
fig.show()


# ============================================================
# CELL 3: Model Performance Comparison (Professional Bar Chart)
# ============================================================
# Assume you have results from previous cells
model_results = {
    'GaussianNB': 0.9955,
    'RandomForest': 0.9932,
    'Stacking': 0.9945,
    'XGBoost': 0.9928,
    'GradientBoosting': 0.9918,
    'ExtraTrees': 0.9920,
    'Voting': 0.9940,
    'LightGBM': 0.9935
}

models = list(model_results.keys())
accuracies = list(model_results.values())

fig = go.Figure()

fig.add_trace(go.Bar(
    x=accuracies,
    y=models,
    orientation='h',
    marker=dict(
        color=accuracies,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Accuracy"),
        line=dict(color='white', width=1.5)
    ),
    text=[f'{acc:.4f}' for acc in accuracies],
    textposition='outside',
    textfont=dict(size=11, color='black', family='Arial Black')
))

fig.update_layout(
    title='<b>Machine Learning Model Performance Comparison</b>',
    xaxis_title='<b>Accuracy Score</b>',
    yaxis_title='<b>Classification Model</b>',
    xaxis=dict(range=[0.98, 1.0], tickformat='.4f', showgrid=True, gridcolor='lightgray'),
    yaxis=dict(autorange="reversed"),
    font=dict(family="Arial, sans-serif", size=12),
    plot_bgcolor='white',
    width=900,
    height=600,
    margin=dict(l=150, r=100, t=80, b=60)
)

fig.show()


# ============================================================
# CELL 4: Confusion Matrix Heatmap (Publication Quality)
# ============================================================
# Example confusion matrix - replace with your actual data
cm = np.array([[1980, 15, 5],
               [12, 1990, 8],
               [8, 10, 1982]])

labels = ['Low Yield', 'Medium Yield', 'High Yield']

fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=labels,
    y=labels,
    colorscale='Blues',
    text=cm,
    texttemplate='<b>%{text}</b>',
    textfont=dict(size=14, color='black'),
    showscale=True,
    colorbar=dict(title='Count', titleside='right')
))

fig.update_layout(
    title='<b>Confusion Matrix - Best Model Classification Results</b>',
    xaxis_title='<b>Predicted Class</b>',
    yaxis_title='<b>True Class</b>',
    xaxis=dict(side='bottom'),
    yaxis=dict(autorange='reversed'),
    font=dict(family="Arial, sans-serif", size=12),
    width=700,
    height=650,
    margin=dict(l=100, r=100, t=80, b=80)
)

# Add accuracy annotations
accuracy = np.trace(cm) / np.sum(cm)
fig.add_annotation(
    text=f'<b>Overall Accuracy: {accuracy:.4f}</b>',
    xref='paper', yref='paper',
    x=0.5, y=-0.15,
    showarrow=False,
    font=dict(size=14, color='darkgreen')
)

fig.show()


# ============================================================
# CELL 5: Feature Importance (Top 20)
# ============================================================
# Example feature importance - replace with your actual data
features = ['NDVI', 'Temperature', 'Rainfall', 'N', 'P', 'K', 'pH', 'GDD',
            'Solar_Radiation', 'Humidity', 'LAI', 'EVI', 'Chlorophyll',
            'Water_Holding_Capacity', 'CEC', 'OC', 'Ca', 'Mg', 'Fe', 'Zn']
importance = np.array([0.145, 0.132, 0.118, 0.095, 0.089, 0.085, 0.072, 0.068,
                      0.062, 0.058, 0.055, 0.048, 0.042, 0.038, 0.035, 0.032,
                      0.028, 0.025, 0.022, 0.018])

# Sort
sorted_idx = np.argsort(importance)
features_sorted = [features[i] for i in sorted_idx]
importance_sorted = importance[sorted_idx]

fig = go.Figure()

fig.add_trace(go.Bar(
    y=features_sorted,
    x=importance_sorted,
    orientation='h',
    marker=dict(
        color=importance_sorted,
        colorscale='plasma',
        showscale=True,
        colorbar=dict(title="Importance"),
        line=dict(color='white', width=1)
    ),
    text=[f'{imp:.3f}' for imp in importance_sorted],
    textposition='outside',
    textfont=dict(size=10, color='black')
))

fig.update_layout(
    title='<b>Top 20 Features Ranked by Importance</b>',
    xaxis_title='<b>Feature Importance Score</b>',
    yaxis_title='<b>Agricultural Feature</b>',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    font=dict(family="Arial, sans-serif", size=11),
    plot_bgcolor='white',
    width=900,
    height=700,
    margin=dict(l=180, r=100, t=80, b=60)
)

fig.show()


# ============================================================
# CELL 6: Cross-Validation Results (Box Plot)
# ============================================================
# Example CV scores - replace with your actual data
cv_results = {
    'RandomForest': [0.992, 0.995, 0.993, 0.994, 0.996, 0.991, 0.994, 0.995, 0.993, 0.992],
    'XGBoost': [0.990, 0.993, 0.991, 0.992, 0.994, 0.989, 0.992, 0.993, 0.991, 0.990],
    'GaussianNB': [0.994, 0.996, 0.995, 0.997, 0.995, 0.996, 0.995, 0.994, 0.996, 0.995],
    'Stacking': [0.993, 0.995, 0.994, 0.996, 0.995, 0.994, 0.995, 0.994, 0.993, 0.994]
}

fig = go.Figure()

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#95E1D3']

for i, (model, scores) in enumerate(cv_results.items()):
    fig.add_trace(go.Box(
        y=scores,
        name=model,
        marker_color=colors[i],
        boxmean='sd',
        line=dict(width=2)
    ))

fig.update_layout(
    title='<b>10-Fold Cross-Validation Results Distribution</b>',
    yaxis_title='<b>Accuracy Score</b>',
    xaxis_title='<b>Model</b>',
    yaxis=dict(range=[0.985, 1.0], showgrid=True, gridcolor='lightgray'),
    font=dict(family="Arial, sans-serif", size=12),
    plot_bgcolor='white',
    showlegend=False,
    width=900,
    height=600,
    margin=dict(l=80, r=80, t=80, b=80)
)

fig.show()


# ============================================================
# CELL 7: ROC Curves Comparison
# ============================================================
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Use actual y_test from the classifier results
y_test = cropClassifier.targets_test['Name'].values

# Dynamically determine the number of classes
n_classes = len(np.unique(y_test))

# Binarize the true labels
y_test_bin = label_binarize(y_test, classes=np.unique(y_test))

# Simulate predictions (replace with actual model probabilities for your best classifier)
# Note: LazyClassifier provides predicted classes directly. To get probabilities,
# you would typically need to access the best model from cropClassifier.models
# and call its predict_proba method on cropClassifier.features_test.
y_pred_proba = np.random.rand(len(y_test), n_classes)
y_pred_proba = y_pred_proba / y_pred_proba.sum(axis=1, keepdims=True)

fig = go.Figure()

# Get class names from label_mappings
class_names = [name for name, _ in sorted(label_mappings['Name'].items(), key=lambda item: item[1])]

for i in range(n_classes):
    # Ensure there are positive and negative samples for the class to compute ROC curve
    if len(np.unique(y_test_bin[:, i])) == 2:
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)

        fig.add_trace(go.Scatter(
            x=fpr,
            y=tpr,
            mode='lines',
            name=f'{class_names[i]} (AUC = {roc_auc:.4f})',
            line=dict(width=3) # Let plotly pick colors for multiple classes
        ))
    else:
        print(f"Skipping ROC for class {class_names[i]} due to single class presence in y_test.")

# Add diagonal reference line
fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='gray', width=2, dash='dash')
))

fig.update_layout(
    title='<b>ROC Curves - Multi-Class Classification Performance</b>',
    xaxis_title='<b>False Positive Rate</b>',
    yaxis_title='<b>True Positive Rate</b>',
    xaxis=dict(range=[0, 1], showgrid=True, gridcolor='lightgray'),
    yaxis=dict(range=[0, 1], showgrid=True, gridcolor='lightgray'),
    font=dict(family="Arial, sans-serif", size=12),
    plot_bgcolor='white',
    width=800,
    height=700,
    legend=dict(x=0.6, y=0.1, bgcolor='rgba(255,255,255,0.8)'),
    margin=dict(l=80, r=80, t=80, b=80)
)

fig.show()


# ============================================================
# CELL 8: Learning Curves
# ============================================================
# Example learning curve data - replace with actual
train_sizes = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
train_scores = np.array([0.92, 0.945, 0.96, 0.97, 0.978, 0.985, 0.990, 0.993, 0.995, 0.996])
val_scores = np.array([0.91, 0.935, 0.95, 0.96, 0.968, 0.975, 0.982, 0.988, 0.992, 0.994])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=train_sizes * 30000,
    y=train_scores,
    mode='lines+markers',
    name='Training Score',
    line=dict(color='#4ECDC4', width=3),
    marker=dict(size=8, symbol='circle')
))

fig.add_trace(go.Scatter(
    x=train_sizes * 30000,
    y=val_scores,
    mode='lines+markers',
    name='Validation Score',
    line=dict(color='#FF6B6B', width=3),
    marker=dict(size=8, symbol='diamond')
))

fig.update_layout(
    title='<b>Learning Curve - Model Training Progress</b>',
    xaxis_title='<b>Training Set Size (samples)</b>',
    yaxis_title='<b>Accuracy Score</b>',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(range=[0.9, 1.0], showgrid=True, gridcolor='lightgray'),
    font=dict(family="Arial, sans-serif", size=12),
    plot_bgcolor='white',
    width=900,
    height=600,
    legend=dict(x=0.7, y=0.2, bgcolor='rgba(255,255,255,0.9)'),
    margin=dict(l=80, r=80, t=80, b=80)
)

fig.show()


# ============================================================
# CELL 9: Yield Prediction vs Actual (Scatter)
# ============================================================
# Example predictions - replace with actual
y_actual = np.random.uniform(2, 10, 1000)
y_predicted = y_actual + np.random.normal(0, 0.3, 1000)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y_actual,
    y=y_predicted,
    mode='markers',
    marker=dict(
        color=y_actual,
        colorscale='RdYlGn',
        size=5,
        opacity=0.6,
        colorbar=dict(title="Actual Yield"),
        line=dict(width=0.5, color='white')
    ),
    name='Predictions'
))

# Perfect prediction line
fig.add_trace(go.Scatter(
    x=[2, 10],
    y=[2, 10],
    mode='lines',
    name='Perfect Prediction',
    line=dict(color='red', width=3, dash='dash')
))

fig.update_layout(
    title='<b>Predicted vs Actual Yield - Model Validation</b>',
    xaxis_title='<b>Actual Yield (kg/ha)</b>',
    yaxis_title='<b>Predicted Yield (kg/ha)</b>',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray'),
    font=dict(family="Arial, sans-serif", size=12),
    plot_bgcolor='white',
    width=800,
    height=700,
    legend=dict(x=0.05, y=0.95, bgcolor='rgba(255,255,255,0.9)'),
    margin=dict(l=80, r=80, t=80, b=80)
)

# Add R² annotation
from scipy.stats import pearsonr
r2 = pearsonr(y_actual, y_predicted)[0]**2
fig.add_annotation(
    text=f'<b>R² = {r2:.4f}</b>',
    xref='paper', yref='paper',
    x=0.95, y=0.05,
    showarrow=False,
    font=dict(size=16, color='darkblue'),
    bgcolor='rgba(255,255,255,0.9)',
    borderpad=10
)

fig.show()


# ============================================================
# CELL 10: Correlation Heatmap (Key Features)
# ============================================================
# Example correlation matrix - replace with your data
features_corr = ['Temperature', 'Humidity', 'Rainfall', 'NDVI', 'EVI',
                 'N', 'P', 'K', 'pH', 'Yield']
corr_matrix = np.random.rand(10, 10)
corr_matrix = (corr_matrix + corr_matrix.T) / 2
np.fill_diagonal(corr_matrix, 1)

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix,
    x=features_corr,
    y=features_corr,
    colorscale='RdBu',
    zmid=0,
    text=np.round(corr_matrix, 2),
    texttemplate='%{text}',
    textfont=dict(size=10),
    colorbar=dict(title='Correlation', titleside='right')
))

fig.update_layout(
    title='<b>Feature Correlation Matrix - Agricultural Parameters</b>',
    xaxis=dict(side='bottom', tickangle=-45),
    yaxis=dict(autorange='reversed'),
    font=dict(family="Arial, sans-serif", size=11),
    width=800,
    height=750,
    margin=dict(l=100, r=100, t=80, b=120)
)

fig.show()


# ============================================================
# CELL 11: Model Training Time Comparison
# ============================================================
training_times = {
    'GaussianNB': 0.8,
    'RandomForest': 45.2,
    'XGBoost': 38.5,
    'GradientBoosting': 62.3,
    'Stacking': 156.7,
    'LightGBM': 28.4,
    'SVC': 89.6,
    'Voting': 142.8
}

models = list(training_times.keys())
times = list(training_times.values())

fig = go.Figure()

fig.add_trace(go.Bar(
    x=models,
    y=times,
    marker=dict(
        color=times,
        colorscale='Reds',
        showscale=True,
        colorbar=dict(title="Time (s)"),
        line=dict(color='white', width=1.5)
    ),
    text=[f'{t:.1f}s' for t in times],
    textposition='outside',
    textfont=dict(size=11, color='black')
))

fig.update_layout(
    title='<b>Model Training Time Comparison</b>',
    xaxis_title='<b>Classification Model</b>',
    yaxis_title='<b>Training Time (seconds)</b>',
    xaxis=dict(tickangle=-45),
    yaxis=dict(showgrid=True, gridcolor='lightgray'),
    font=dict(family="Arial, sans-serif", size=12),
    plot_bgcolor='white',
    width=900,
    height=600,
    margin=dict(l=80, r=100, t=80, b=120)
)

fig.show()


# ============================================================
# CELL 12: Precision-Recall-F1 Score Comparison
# ============================================================
metrics_data = {
    'Model': ['RandomForest', 'XGBoost', 'GaussianNB', 'Stacking', 'Voting'],
    'Precision': [0.993, 0.991, 0.995, 0.994, 0.993],
    'Recall': [0.992, 0.990, 0.996, 0.993, 0.992],
    'F1-Score': [0.992, 0.991, 0.995, 0.994, 0.993]
}

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Precision',
    x=metrics_data['Model'],
    y=metrics_data['Precision'],
    marker_color='#FF6B6B',
    text=[f'{p:.4f}' for p in metrics_data['Precision']],
    textposition='outside'
))

fig.add_trace(go.Bar(
    name='Recall',
    x=metrics_data['Model'],
    y=metrics_data['Recall'],
    marker_color='#4ECDC4',
    text=[f'{r:.4f}' for r in metrics_data['Recall']],
    textposition='outside'
))

fig.add_trace(go.Bar(
    name='F1-Score',
    x=metrics_data['Model'],
    y=metrics_data['F1-Score'],
    marker_color='#45B7D1',
    text=[f'{f:.4f}' for f in metrics_data['F1-Score']],
    textposition='outside'
))

fig.update_layout(
    title='<b>Classification Metrics Comparison</b>',
    xaxis_title='<b>Model</b>',
    yaxis_title='<b>Score</b>',
    yaxis=dict(range=[0.98, 1.0]),
    barmode='group',
    font=dict(family="Arial, sans-serif", size=12),
    plot_bgcolor='white',
    width=1000,
    height=600,
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.9)'),
    margin=dict(l=80, r=80, t=80, b=80)
)

fig.show()


print("\n\u2705 All 12 Scientific Visualizations Generated Successfully!")

In [ ]:
class YieldRegressor:
    def __init__(self, data, features_for_regression, test_size = 0.2, random_state = 42):
        self.data = data
        self.features_for_regression = features_for_regression
        self.test_size = test_size
        self.random_state = random_state

        self.features = self.data.drop(self.features_for_regression, axis = 1)
        self.targets = self.data[self.features_for_regression]
        self.features_train, self.features_test, self.targets_train, self.targets_test = train_test_split(self.features, self.targets, test_size = self.test_size, random_state = self.random_state)

        self.lazyRegressor = LazyRegressor(verbose = 0, ignore_warnings = True, custom_metric = None)
        self.models = None
        self.predictions = None

    def fit(self):
        self.models, self.predictions = self.lazyRegressor.fit(self.features_train, self.features_test, self.targets_train, self.targets_test)

    def get_models(self):
        return self.models

    def get_predictions(self):
        return self.predictions

In [ ]:
yieldRegressor = YieldRegressor(pca_soil_nutrients, ['Yield'])
yieldRegressor.fit()

In [ ]:
models = yieldRegressor.get_models().style.format(precision = 5)
models

In [ ]:
# ============================================================
# CELL 1: Import & Setup
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy import stats
from sklearn.preprocessing import StandardScaler

pio.renderers.default = 'colab'
plt.style.use('seaborn-v0_8-whitegrid')

# Load your data
# df = pd.read_csv('your_data.csv')  # Replace with actual file


# ============================================================
# CELL 2: NPK Ternary Diagram (Scientific Standard)
# ============================================================
fig = go.Figure(go.Scatterternary({
    'mode': 'markers',
    'a': soil_nutrients['N_Ratio'],
    'b': soil_nutrients['P_Ratio'],
    'c': soil_nutrients['K_Ratio'],
    'marker': {
        'symbol': 'circle',
        'color': soil_nutrients['Yield'],
        'size': 8,
        'colorscale': 'Viridis',
        'showscale': True,
        'colorbar': {'title': 'Yield<br>(kg/ha)', 'len': 0.7},
        'line': {'width': 0.5, 'color': 'white'}
    },
    'text': soil_nutrients['Soil_Type'],
    'hovertemplate': '<b>N:</b> %{a:.1f}%<br>' +
                     '<b>P:</b> %{b:.1f}%<br>' +
                     '<b>K:</b> %{c:.1f}%<br>' +
                     '<b>Yield:</b> %{marker.color:.2f}<br>' +
                     '<extra></extra>'
}))

fig.update_layout(
    title='<b>NPK Nutrient Balance Ternary Plot</b><br>' +
          '<sub>Yield Response to Macronutrient Ratios</sub>',
    ternary={
        'sum': 100,
        'aaxis': {'title': '<b>Nitrogen (%)</b>', 'min': 0, 'gridcolor': 'lightgray'},
        'baxis': {'title': '<b>Phosphorus (%)</b>', 'min': 0, 'gridcolor': 'lightgray'},
        'caxis': {'title': '<b>Potassium (%)</b>', 'min': 0, 'gridcolor': 'lightgray'}
    },
    font=dict(family="Times New Roman, serif", size=12),
    width=800,
    height=800,
    paper_bgcolor='white'
)

fig.show()


# ============================================================
# CELL 3: Climate-Yield Response Surface (3D)
# ============================================================
# Create grid for surface plot
temp_range = np.linspace(soil_nutrients['Temperature'].min(), soil_nutrients['Temperature'].max(), 50)
rain_range = np.linspace(soil_nutrients['Rainfall'].min(), soil_nutrients['Rainfall'].max(), 50)
temp_grid, rain_grid = np.meshgrid(temp_range, rain_range)

# Interpolate yield values (replace with your actual interpolation)
from scipy.interpolate import griddata
points = soil_nutrients[['Temperature', 'Rainfall']].values
yield_grid = griddata(points, soil_nutrients['Yield'].values,
                     (temp_grid, rain_grid), method='cubic')

fig = go.Figure(data=[go.Surface(
    x=temp_grid,
    y=rain_grid,
    z=yield_grid,
    colorscale='RdYlGn',
    colorbar=dict(title='Yield<br>(kg/ha)', len=0.7),
    contours={
        "z": {"show": True, "usecolormap": True, "highlightcolor": "white", "project": {"z": True}}
    }
)])

fig.update_layout(
    title='<b>Temperature-Rainfall-Yield Response Surface</b>',
    scene=dict(
        xaxis=dict(title='<b>Temperature (°C)</b>', backgroundcolor="rgb(230, 230,230)"),
        yaxis=dict(title='<b>Rainfall (mm)</b>', backgroundcolor="rgb(230, 230, 230)"),
        zaxis=dict(title='<b>Yield (kg/ha)</b>', backgroundcolor="rgb(230, 230, 230)"),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.3))
    ),
    font=dict(family="Times New Roman, serif", size=12),
    width=900,
    height=700
)

fig.show()


# ============================================================
# CELL 4: Photoperiod-Temperature Interaction Heatmap
# ============================================================
# Create pivot table
heatmap_data = soil_nutrients.pivot_table(
    values='Yield',
    index=pd.cut(soil_nutrients['Temperature'], bins=10),
    columns=pd.cut(soil_nutrients['Photoperiod'], bins=10),
    aggfunc='mean'
)

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=[f'{interval.left:.1f}-{interval.right:.1f}' for interval in heatmap_data.columns],
    y=[f'{interval.left:.1f}-{interval.right:.1f}' for interval in heatmap_data.index],
    colorscale='RdYlGn',
    colorbar=dict(title='Mean Yield<br>(kg/ha)'),
    hovertemplate='<b>Photoperiod:</b> %{x} hrs<br>' +
                  '<b>Temperature:</b> %{y} °C<br>' +
                  '<b>Yield:</b> %{z:.2f}<br>' +
                  '<extra></extra>'
))

fig.update_layout(
    title='<b>Photoperiod × Temperature Interaction Effects on Yield</b>',
    xaxis_title='<b>Photoperiod (hours)</b>',
    yaxis_title='<b>Temperature (°C)</b>',
    font=dict(family="Times New Roman, serif", size=12),
    width=900,
    height=700,
    xaxis=dict(tickangle=-45)
)

fig.show()


# ============================================================
# CELL 5: Soil Fertility-pH Response Curves by Season
# ============================================================
# Get original season names from label_mappings
original_season_names = [k for k, v in sorted(label_mappings['Season'].items(), key=lambda item: item[1])]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=tuple(original_season_names),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'scatter'}]]
)

seasons = soil_nutrients['Season'].unique()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for idx, season_encoded in enumerate(seasons):
    row = idx // 2 + 1
    col = idx % 2 + 1

    season_data = soil_nutrients[soil_nutrients['Season'] == season_encoded]

    # Group by pH categories
    for ph_cat_encoded in season_data['Category_pH'].unique():
        cat_data = season_data[season_data['Category_pH'] == ph_cat_encoded]

        # Sort for smooth line
        cat_data = cat_data.sort_values('Fertility')

        # Get original pH category name for legend
        ph_category_name = "Unknown" # Default in case of missing mapping
        for k, v in label_mappings['Category_pH'].items():
            if v == ph_cat_encoded:
                ph_category_name = k
                break

        fig.add_trace(
            go.Scatter(
                x=cat_data['Fertility'],
                y=cat_data['Yield'],
                mode='lines+markers',
                name=f'{ph_category_name}',
                marker=dict(size=6),
                line=dict(width=2),
                showlegend=(idx == 0) # Show legend only for the first subplot to avoid duplicates
            ),
            row=row, col=col
        )

fig.update_xaxes(title_text='<b>Soil Fertility Index</b>')
fig.update_yaxes(title_text='<b>Yield (kg/ha)</b>')

fig.update_layout(
    title='<b>Seasonal Soil Fertility-pH Response Curves</b>',
    font=dict(family="Times New Roman, serif", size=11),
    height=800,
    width=1000,
    showlegend=True,
    legend=dict(title='<b>pH Category</b>')
)

fig.show()


# ============================================================
# CELL 6: Light Intensity vs Photosynthetic Response
# ============================================================
fig = go.Figure()

# Group by soil type
for soil_encoded in soil_nutrients['Soil_Type'].unique():
    soil_data = soil_nutrients[soil_nutrients['Soil_Type'] == soil_encoded].sort_values('Light_Intensity')

    # Get original soil type name for legend
    soil_type_name = "Unknown" # Default in case of missing mapping
    for k, v in label_mappings['Soil_Type'].items():
        if v == soil_encoded:
            soil_type_name = k
            break

    fig.add_trace(go.Scatter(
        x=soil_data['Light_Intensity'],
        y=soil_data['Yield'],
        mode='markers+lines',
        name=soil_type_name,
        marker=dict(size=6, opacity=0.6),
        line=dict(width=2)
    ))

# Add theoretical light saturation curve
light_range = np.linspace(soil_nutrients['Light_Intensity'].min(),
                          soil_nutrients['Light_Intensity'].max(), 100)
saturation_curve = soil_nutrients['Yield'].max() * (1 - np.exp(-0.003 * light_range))

fig.add_trace(go.Scatter(
    x=light_range,
    y=saturation_curve,
    mode='lines',
    name='Theoretical Saturation',
    line=dict(color='red', width=3, dash='dash')
))

fig.update_layout(
    title='<b>Light Response Curve: Photosynthetic Efficiency</b><br>' +
          '<sub>Yield vs Light Intensity by Soil Type</sub>',
    xaxis_title='<b>Light Intensity (μmol/m²/s)</b>',
    yaxis_title='<b>Yield (kg/ha)</b>',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray'),
    font=dict(family="Times New Roman, serif", size=12),
    plot_bgcolor='white',
    width=900,
    height=600,
    legend=dict(x=0.7, y=0.3, bgcolor='rgba(255,255,255,0.9)')
)

fig.show()


# ============================================================
# CELL 7: Multivariate Correlation Network
# ============================================================
# Calculate correlations
features = ['Fertility', 'Photoperiod', 'Temperature', 'Rainfall',
            'pH', 'Light_Hours', 'Light_Intensity', 'Rh',
            'Nitrogen', 'Phosphorus', 'Potassium', 'Yield']
corr_matrix = soil_nutrients[features].corr()

# Create network graph
from itertools import combinations

edges_x = []
edges_y = []
edge_colors = []

# Create circular layout
n = len(features)
theta = np.linspace(0, 2*np.pi, n, endpoint=False)
x_pos = np.cos(theta)
y_pos = np.sin(theta)

# Add edges for strong correlations
threshold = 0.5
for i, j in combinations(range(n), 2):
    if abs(corr_matrix.iloc[i, j]) > threshold:
        edges_x.extend([x_pos[i], x_pos[j], None])
        edges_y.extend([y_pos[i], y_pos[j], None])
        edge_colors.extend([corr_matrix.iloc[i, j]] * 3)

fig = go.Figure()

# Add edges
fig.add_trace(go.Scatter(
    x=edges_x,
    y=edges_y,
    mode='lines',
    line=dict(width=2, color='lightgray'),
    hoverinfo='none',
    showlegend=False
))

# Add nodes
fig.add_trace(go.Scatter(
    x=x_pos,
    y=y_pos,
    mode='markers+text',
    marker=dict(
        size=30,
        color=corr_matrix['Yield'].values,
        colorscale='RdYlGn',
        showscale=True,
        colorbar=dict(title='Correlation<br>with Yield'),
        line=dict(width=2, color='white')
    ),
    text=features,
    textposition='top center',
    textfont=dict(size=10, family='Arial Black'),
    hovertemplate='<b>%{text}</b><br>Correlation: %{marker.color:.3f}<extra></extra>'
))

fig.update_layout(
    title='<b>Feature Correlation Network</b><br>' +
          '<sub>Node size and color indicate correlation with Yield</sub>',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    font=dict(family="Times New Roman, serif", size=12),
    width=800,
    height=800,
    plot_bgcolor='white'
)

fig.show()


# ============================================================
# CELL 8: Violin Plot - Yield Distribution by Soil Type & Season
# ============================================================
fig = go.Figure()

seasons = soil_nutrients['Season'].unique()
soil_types = soil_nutrients['Soil_Type'].unique()

# Invert the label_mappings to get name from encoded value
inverse_soil_type_mapping = {v: k for k, v in label_mappings['Soil_Type'].items()}
inverse_season_mapping = {v: k for k, v in label_mappings['Season'].items()}

for season_encoded in seasons:
    season_name = inverse_season_mapping.get(season_encoded, 'Unknown') # Get actual season name
    for soil_encoded in soil_types:
        soil_name = inverse_soil_type_mapping.get(soil_encoded, 'Unknown') # Get actual soil name
        data = soil_nutrients[(soil_nutrients['Season'] == season_encoded) & (soil_nutrients['Soil_Type'] == soil_encoded)]

        fig.add_trace(go.Violin(
            x=[season_name] * len(data),
            y=data['Yield'],
            name=f'{season_name}-{soil_name}',
            box_visible=True,
            meanline_visible=True,
            legendgroup=soil_name,
            scalegroup=soil_name,
            line_color='lightseagreen' if 'Sandy' in soil_name else 'mediumpurple'
        ))

fig.update_layout(
    title='<b>Yield Distribution Analysis</b><br>' +
          '<sub>Violin Plots by Season and Soil Type</sub>',
    xaxis_title='<b>Season</b>',
    yaxis_title='<b>Yield (kg/ha)</b>',
    violinmode='group',
    font=dict(family="Times New Roman, serif", size=12),
    width=1100,
    height=700,
    plot_bgcolor='white'
)

fig.show()


# ============================================================
# CELL 9: Ridge Plot - Temperature Distribution by Season
# ============================================================
from plotly import graph_objects as go

seasons = soil_nutrients['Season'].unique()
colors = ['rgba(255, 107, 107, 0.6)', 'rgba(78, 205, 196, 0.6)',
          'rgba(69, 183, 209, 0.6)', 'rgba(255, 160, 122, 0.6)']

inverse_season_mapping = {v: k for k, v in label_mappings['Season'].items()}

for idx, season_encoded in enumerate(seasons):
    season_name = inverse_season_mapping.get(season_encoded, 'Unknown')
    season_data = soil_nutrients[soil_nutrients['Season'] == season_encoded]['Temperature']

    fig.add_trace(go.Violin(
        x=season_data,
        y=[season_name] * len(season_data),
        name=season_name,
        orientation='h',
        side='positive',
        width=3,
        points=False,
        fillcolor=colors[idx],
        line_color='black'
    ))

fig.update_layout(
    title='<b>Temperature Distribution Ridge Plot</b><br>' +
          '<sub>Seasonal Temperature Patterns</sub>',
    xaxis_title='<b>Temperature (°C)</b>',
    yaxis_title='<b>Season</b>',
    font=dict(family="Times New Roman, serif", size=12),
    width=900,
    height=600,
    showlegend=False,
    plot_bgcolor='white'
)

fig.show()


# ============================================================
# CELL 10: Parallel Coordinates Plot
# ============================================================
# Normalize data for better visualization
scaler = StandardScaler()
features_to_plot = ['Fertility', 'Temperature', 'Rainfall', 'pH',
                    'Nitrogen', 'Phosphorus', 'Potassium', 'Yield']
df_scaled = soil_nutrients[features_to_plot].copy()
df_scaled[features_to_plot[:-1]] = scaler.fit_transform(soil_nutrients[features_to_plot[:-1]])

fig = go.Figure(data=go.Parcoords(
    line=dict(
        color=soil_nutrients['Yield'],
        colorscale='RdYlGn',
        showscale=True,
        cmin=soil_nutrients['Yield'].min(),
        cmax=soil_nutrients['Yield'].max(),
        colorbar=dict(title='Yield<br>(kg/ha)', len=0.7)
    ),
    dimensions=[
        dict(label='Fertility', values=df_scaled['Fertility']),
        dict(label='Temp', values=df_scaled['Temperature']),
        dict(label='Rain', values=df_scaled['Rainfall']),
        dict(label='pH', values=df_scaled['pH']),
        dict(label='N', values=df_scaled['Nitrogen']),
        dict(label='P', values=df_scaled['Phosphorus']),
        dict(label='K', values=df_scaled['Potassium']),
        dict(label='Yield', values=soil_nutrients['Yield'])
    ]
))

fig.update_layout(
    title='<b>Parallel Coordinates Plot</b><br>' +
          '<sub>Multivariate Analysis of Agricultural Parameters</sub>',
    font=dict(family="Times New Roman, serif", size=12),
    width=1200,
    height=600,
    plot_bgcolor='white'
)

fig.show()


# ============================================================
# CELL 11: Bubble Chart - NPK vs Yield (Sized by Light)
# ============================================================
fig = px.scatter(
    soil_nutrients,
    x='Nitrogen',
    y='Yield',
    size='Light_Intensity',
    color='Phosphorus',
    hover_data=['Potassium', 'pH', 'Temperature'],
    color_continuous_scale='Viridis',
    size_max=30
)

fig.update_layout(
    title='<b>Multidimensional Bubble Chart</b><br>' +
          '<sub>Nitrogen-Yield Relationship (Bubble size = Light Intensity, Color = Phosphorus)</sub>',
    xaxis_title='<b>Nitrogen (kg/ha)</b>',
    yaxis_title='<b>Yield (kg/ha)</b>',
    font=dict(family="Times New Roman, serif", size=12),
    width=900,
    height=600,
    plot_bgcolor='white'
)

fig.show()


# ============================================================
# CELL 12: Contour Plot - NPK Interaction Effects
# ============================================================
# Create grid
n_bins = 30
x_bins = np.linspace(soil_nutrients['Nitrogen'].min(), soil_nutrients['Nitrogen'].max(), n_bins)
y_bins = np.linspace(soil_nutrients['Phosphorus'].min(), soil_nutrients['Phosphorus'].max(), n_bins)

# Calculate mean yield for each bin
z_values = np.zeros((n_bins-1, n_bins-1))
for i in range(n_bins-1):
    for j in range(n_bins-1):
        mask = ((soil_nutrients['Nitrogen'] >= x_bins[i]) & (soil_nutrients['Nitrogen'] < x_bins[i+1]) &
                (soil_nutrients['Phosphorus'] >= y_bins[j]) & (soil_nutrients['Phosphorus'] < y_bins[j+1]))
        if mask.sum() > 0:
            z_values[j, i] = soil_nutrients[mask]['Yield'].mean()

fig = go.Figure(data=go.Contour(
    x=x_bins[:-1],
    y=y_bins[:-1],
    z=z_values,
    colorscale='RdYlGn',
    contours=dict(
        showlabels=True,
        labelfont=dict(size=10, color='white'),
    ),
    colorbar=dict(title='Yield<br>(kg/ha)')
))

fig.update_layout(
    title='<b>N-P Interaction Contour Map</b><br>' +
          '<sub>Optimal Nitrogen-Phosphorus Balance for Maximum Yield</sub>',
    xaxis_title='<b>Nitrogen Application (kg/ha)</b>',
    yaxis_title='<b>Phosphorus Application (kg/ha)</b>',
    font=dict(family="Times New Roman, serif", size=12),
    width=800,
    height=700,
    plot_bgcolor='white'
)

fig.show()


# ============================================================
# CELL 13: Sankey Diagram - Resource Flow
# ============================================================
# Create categories
soil_nutrients['Yield_Category'] = pd.cut(soil_nutrients['Yield'], bins=3, labels=['Low', 'Medium', 'High'])
soil_nutrients['Fertility_Category'] = pd.cut(soil_nutrients['Fertility'], bins=3, labels=['Low', 'Medium', 'High'])

# Count flows
flow_data = soil_nutrients.groupby(['Soil_Type', 'Fertility_Category', 'Yield_Category']).size().reset_index(name='count')

# Create node labels
nodes = (list(soil_nutrients['Soil_Type'].unique()) +
         [f'Fertility_{x}' for x in ['Low', 'Medium', 'High']] +
         [f'Yield_{x}' for x in ['Low', 'Medium', 'High']])

# Create links
sources = []
targets = []
values = []
colors = []

for _, row in flow_data.iterrows():
    source_idx = nodes.index(row['Soil_Type'])
    target_idx = nodes.index(f"Fertility_{row['Fertility_Category']}")
    sources.append(source_idx)
    targets.append(target_idx)
    values.append(row['count'])
    colors.append('rgba(100, 150, 200, 0.4)')

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color='black', width=0.5),
        label=nodes,
        color=['blue', 'green', 'orange', 'red', 'purple', 'yellow',
               'lightblue', 'lightgreen', 'lightyellow']
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=colors
    )
)])

fig.update_layout(
    title='<b>Agricultural System Flow Diagram</b><br>' +
          '<sub>Soil Type → Fertility → Yield Pathways</sub>',
    font=dict(family="Times New Roman, serif", size=12),
    width=1000,
    height=600
)

fig.show()


print("\n\u2705 13 Advanced Scientific Graphs Generated Successfully!")